# WHO guidelines compliance

- Inputs: `data/cleaned/gdd_country_year.csv`, `data/reference/who_guidelines.csv`, `data/reference/countries.csv`
- Output: `data/cleaned/tab_guidelines.csv`, one row per country x year x guideline
- The guidelines table has no iso3. It is a lookup joined on `nutrient`, so every country-year picks up the same min/max.
- Total fat and trans fat are not in GDD, so they are left out.
- Free sugars use GDD added sugars (v35), which leaves out sugar from fruit juice, so it underestimates free sugars.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from check_schema import check_schema

REF = Path('../data/reference')
CLEANED = Path('../data/cleaned')

diet = pd.read_csv(CLEANED / 'gdd_country_year.csv')
guidelines = pd.read_csv(REF / 'who_guidelines.csv')
countries = pd.read_csv(REF / 'countries.csv')

Step 1. Convert GDD codes into the guideline units, naming each column after the `nutrient` in `who_guidelines.csv`.

In [2]:
conv = pd.DataFrame({
    'iso3': diet.iso3,
    'year': diet.year,
    'carbohydrate':  diet.v22,                  # % of calories
    'free_sugars':   diet.v35,                  # added sugars, % of calories
    'saturated_fat': diet.v27,                  # % of calories
    'protein':       diet.v23 * 4 / 2000 * 100, # g/day -> % of calories (GDD assumes 2,000 kcal)
    'salt':          diet.v37 * 2.5 / 1000,     # sodium mg/day -> salt g/day
    'potassium':     diet.v41,                  # mg/day
    'fruit_veg':     diet.v01 + diet.v02,       # fruits + non-starchy vegetables, g/day
    'fibre':         diet.v34,                  # g/day
})

Step 2. Reshape to one row per country x year x nutrient, then join the guidelines on `nutrient`.

In [3]:
long = conv.melt(id_vars=['iso3', 'year'], var_name='nutrient', value_name='value')
df = long.merge(guidelines[['nutrient', 'min', 'max', 'unit']], on='nutrient',
                how='left', validate='many_to_one')

# A spelling mismatch would leave min, max and unit empty and count as 'meets'
assert df['unit'].notna().all(), df.loc[df['unit'].isna(), 'nutrient'].unique()

Step 3. Work out whether each value meets its guideline.

- `meets`: 1 if inside the limits, else 0. An empty min or max means no limit on that side.
- `gap`: how far outside the limit, in the guideline's unit. Negative = below a minimum, positive = above a maximum, 0 = meets.
- `gap_pct`: the gap as a % of the limit it missed, so different units can be compared.

In [4]:
below = df['min'].notna() & (df['value'] < df['min'])
above = df['max'].notna() & (df['value'] > df['max'])

df['meets'] = (~below & ~above).astype(int)
df['gap'] = np.select([below, above], [df['value'] - df['min'], df['value'] - df['max']], 0)
df['gap_pct'] = np.select([below, above], [df['gap'] / df['min'] * 100, df['gap'] / df['max'] * 100], 0)

Step 4. Add display labels and country names, then check and save.

In [5]:
labels = {
    'carbohydrate':  'Carbohydrate (% of calories)',
    'free_sugars':   'Free sugars (% of calories)',
    'saturated_fat': 'Saturated fat (% of calories)',
    'protein':       'Protein (% of calories)',
    'salt':          'Salt (g/day)',
    'potassium':     'Potassium (mg/day)',
    'fruit_veg':     'Fruit and vegetables (g/day)',
    'fibre':         'Fibre (g/day)',
}
df['guideline_label'] = df['nutrient'].map(labels)

df = df.merge(countries.rename(columns={'who_region': 'region'}), on='iso3',
              how='left', validate='many_to_one')
assert df['region'].notna().all(), df.loc[df['region'].isna(), 'iso3'].unique()

df = df[['iso3', 'country_name', 'region', 'year', 'nutrient', 'guideline_label', 'unit',
         'value', 'min', 'max', 'meets', 'gap', 'gap_pct']]
df = df.sort_values(['iso3', 'year', 'nutrient']).reset_index(drop=True)

check_schema(df, 'tab_guidelines', grain=('iso3', 'year', 'nutrient'))
df.to_csv(CLEANED / 'tab_guidelines.csv', index=False)

tab_guidelines: 10360 rows x 13 cols | iso3: 185 | years: 1990-2018
  OK


Sanity check: share of countries meeting each guideline in 2018.

In [6]:
df[df['year'] == 2018].groupby('guideline_label')['meets'].mean().mul(100).round(0).sort_values()

guideline_label
Fruit and vegetables (g/day)      9.0
Potassium (mg/day)                9.0
Salt (g/day)                     22.0
Fibre (g/day)                    30.0
Saturated fat (% of calories)    38.0
Free sugars (% of calories)      48.0
Protein (% of calories)          48.0
Carbohydrate (% of calories)     96.0
Name: meets, dtype: float64